# PH-GPS Analysis Notebook

Analysis for: **"The Pulmonary Hypertension Global Patient Survey: understanding the experiences and perspectives of patients"**  
Published in ERJ Open Research 2025; 11: 00297-2025

**Data**: Cut 1 dataset (October 2023 – February 2024)

This notebook is structured to follow the Results section of the paper. Each section header references the corresponding paper section, table, or figure.

# Setup

In [ ]:
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots


df = pl.read_csv(Path("./data/complete_cleaned_df.csv"))
adult_df = pl.read_csv(Path("./data/complete_cleaned_df_adults.csv"))
child_df = pl.read_csv(Path("./data/complete_cleaned_df_paeds.csv"))


pl.enable_string_cache()
pl.Config.set_fmt_str_lengths(900)
pl.Config.set_tbl_width_chars(900)


In [ ]:
def struct_to_str(data, col) -> str:
    return ",\n".join(
        f"{item[col]}: {item['count']} "
        f"({round(item['count'] / sum([_['count'] for _ in data]) * 100, 1)}%)"
        for item in data
    )

def num_w_percentage_denom(num: float, denom: float, alias: str) -> str:
    return pl.format("{} ({}%)", num, (num / denom * 100).round(1)).alias(alias)

def num_w_percentage_expr(expr: pl.Expr, alias: str) -> str:
    return pl.format("{} ({}%)", expr.sum(), (expr.sum() / expr.len() * 100).round(1)).alias(alias)

def num_w_percentage_float(num: float, denom: float) -> str:
    return f"{num} ({round(num / denom * 100, 1)}%)"


## Medication classification definitions

Used later by PH group 1 and 4 specific analyses (Supp. Tables S16–S18).

In [ ]:
# Drug classification definitions
ph_medical_therapy_vasodilators = [
    "Ambrisentan (Letairis)", "Bosentan (Tracleer)", "Macitentan (Opsumit)", "Sildenafil (Revatio)", "Tadalafil (Cialis)", "Riociguat (Adempas)",
    "Epoprostenol (Flolan)", "Epoprostenol (Veletri)", "Iloprost (Ventavis)", "Selexipag (Uptravi)", "Treprostinil (Remodulin)", "Treprostinil (Tyvaso)",
    "Treprostinil (Orenitram)", "Beraprost", "Sotatercept ", "Ralinepag", "Seralutinib",
]
calcium_channel_blocking_drugs = r"Amlodipine \(Istin\)|Diltiazem \(Adizem, Angitil, Dilcardia, Retalzem, Slozem, Tildiem, Uard, Viazem, Zemtard\)|Nifedipine \(Adalat, Adanif, Adipine, Coracten, Dexipress, Fortipine, Neozipine, Nidef, Nifedipress, Tensipine, Valni\)|Nircadipine \(Cardene\)"

parenteral_drugs = ["Epoprostenol (Flolan)", "Epoprostenol (Veletri)", "Treprostinil (Remodulin)", "Sotatercept"]
unlicensed_drugs = ["sotatercept", "ralinepag", "seralutinib"]
antiplatelet_drugs = ["aspirin", "clopidogrel"]

def get_antiplatelets(col: pl.col) -> pl.Expr:
    return col.str.to_lowercase().str.replace(", ", ";").str.replace(",", ";").str.split(";").list.set_intersection(antiplatelet_drugs).list.len()

def get_parenteral(col: pl.col) -> pl.Expr:
    return col.str.split(";").list.set_intersection(parenteral_drugs).list.len()

def get_num_of_vasodilators(col_1: pl.col, col_2: pl.col) -> pl.Expr:
    return col_1.str.split(";").list.set_union(col_2.str.split(";")).list.set_intersection(ph_medical_therapy_vasodilators).list.len()


# 1. Overview of patients and geographic representation

Paper p3 and Table 1. Demographics of adult and paediatric respondents by geographic region.

## Table 1: Adult patient demographics by region

Paper Table 1, adult rows: respondents, countries, age, sex, PH diagnosis by region.

In [ ]:
num_adult_adult_proxy_exc_dropouts = len(adult_df)
print("Number of adults and adult proxy excluding immediate dropouts: ", num_adult_adult_proxy_exc_dropouts)

adult_df.group_by("region").agg(
    pl.format("{} ({}%)", pl.len(), (pl.len()/num_adult_adult_proxy_exc_dropouts * 100).round(1)).alias("Number of responders"),
    pl.col("country").unique().count().alias("Number of countries represented"),
    pl.format("{} ({})", pl.col("age").mean().round(1), pl.col("age").std().round(1)).alias("Mean age (std.)"),
    pl.format("{} ({})", pl.col("age").quantile(0.5), (pl.col("age").quantile(0.75) - pl.col("age").quantile(0.25))).alias("Median age (IQR)"),
    pl.col("sexAtBirth").value_counts().map_elements(lambda x: struct_to_str(x, "sexAtBirth")).alias("Sex"),
    pl.col("diagnosisClassification (cleaned)").value_counts().map_elements( lambda x: struct_to_str(x, "diagnosisClassification (cleaned)")).alias("PH Diagnosis"),
).sort(by="region").vstack(
    pl.DataFrame({
        "region": ["Total"],
        "Number of responders": [f"{num_adult_adult_proxy_exc_dropouts} (100%)"],
        "Number of countries represented": [len(adult_df["country"].unique())],
        "Mean age (std.)": [f"{round(adult_df['age'].mean(), 1)} ({round(adult_df['age'].std(), 1)})"],
        "Median age (IQR)": [f"{adult_df['age'].quantile(0.5)} ({adult_df['age'].quantile(0.75) - adult_df['age'].quantile(0.25)})"],
        "Sex": [struct_to_str(adult_df["sexAtBirth"].value_counts().sort(by="sexAtBirth").to_struct(), "sexAtBirth")],
        "PH Diagnosis": [struct_to_str(adult_df["diagnosisClassification (cleaned)"].value_counts().sort(by="diagnosisClassification (cleaned)").to_struct(), "diagnosisClassification (cleaned)")],
    }, schema_overrides={"Number of countries represented": pl.UInt32})
)


## Table 1: Group 1 PAH subgroup demographics

Paper Table 1 subset. iPAH/hPAH/CTD-PAH/CHD-PAH subclassification by region.

In [ ]:
adult_exc_dropouts_group_1 = adult_df.filter(
    pl.col("diagnosisClassification (cleaned)") == "Group 1 - pulmonary arterial hypertension (PAH)",
)

num_adult_adult_proxy_exc_dropouts_group_1 = len(adult_exc_dropouts_group_1)
print("Number of adults and adult proxy with Group 1 PH excluding immediate dropouts: ", num_adult_adult_proxy_exc_dropouts_group_1)

adult_exc_dropouts_group_1.group_by("region").agg(
    pl.format("{} ({}%)", pl.len(), (pl.len()/num_adult_adult_proxy_exc_dropouts_group_1 * 100).round(1)).alias("Number of responders"),
    pl.col("country").unique().count().alias("Number of countries represented"),
    pl.format("{} ({})", pl.col("age").mean().round(1), pl.col("age").std().round(1)).alias("Mean age (std.)"),
    pl.format("{} ({})", pl.col("age").quantile(0.5), (pl.col("age").quantile(0.75) - pl.col("age").quantile(0.25))).alias("Median age (IQR)"),
    pl.col("sexAtBirth").value_counts().map_elements(lambda x: struct_to_str(x, "sexAtBirth")).alias("Sex"),
    pl.col("groupOneSubclassification (cleaned)").value_counts().map_elements(lambda x: struct_to_str(x, "groupOneSubclassification (cleaned)")).alias("PH Diagnosis"),
).sort(by="region").vstack(
    pl.DataFrame({
        "region": ["Total"],
        "Number of responders": [f"{num_adult_adult_proxy_exc_dropouts_group_1} (100%)"],
        "Number of countries represented": [len(adult_exc_dropouts_group_1["country"].unique())],
        "Mean age (std.)": [f"{round(adult_exc_dropouts_group_1['age'].mean(), 1)} ({round(adult_exc_dropouts_group_1['age'].std(), 1)})"],
        "Median age (IQR)": [f"{adult_exc_dropouts_group_1['age'].quantile(0.5)} ({adult_exc_dropouts_group_1['age'].quantile(0.75) - adult_exc_dropouts_group_1['age'].quantile(0.25)})"],
        "Sex": [struct_to_str(adult_exc_dropouts_group_1["sexAtBirth"].value_counts().sort(by="sexAtBirth").to_struct(), "sexAtBirth")],
        "PH Diagnosis": [struct_to_str(adult_exc_dropouts_group_1["groupOneSubclassification (cleaned)"].value_counts().sort(by="groupOneSubclassification (cleaned)").to_struct(), "groupOneSubclassification (cleaned)")],
    }, schema_overrides={"Number of countries represented": pl.UInt32})
)

## Table 1: Paediatric respondents

Paper Table 1, paediatric rows.

In [ ]:
num_child_exc_dropouts = len(child_df)
print("Number of paediatric respondents excluding immediate dropouts: ", num_child_exc_dropouts)

child_df.group_by("region").agg(
    pl.format("{} ({}%)", pl.len(), (pl.len()/num_child_exc_dropouts * 100).round(1)).alias("Number of responders"),
    pl.format("{} ({})", pl.col("age").mean().round(1), pl.col("age").std().round(1)).alias("Mean age (std.)"),
    pl.format("{} ({})", pl.col("age").quantile(0.5), (pl.col("age").quantile(0.75) - pl.col("age").quantile(0.25))).alias("Median age (IQR)"),
    pl.col("sexAtBirth").value_counts().map_elements(lambda x: struct_to_str(x, "sexAtBirth")).alias("Sex"),
    pl.col("diagnosisClassification (cleaned)").value_counts().map_elements(lambda x: struct_to_str(x, "diagnosisClassification (cleaned)")).alias("PH Diagnosis"),
).sort(by="region").vstack(
    pl.DataFrame({
        "region": ["Total"],
        "Number of responders": [f"{num_child_exc_dropouts} (100%)"],
        "Mean age (std.)": [f"{round(child_df['age'].mean(), 1)} ({round(child_df['age'].std(), 1)})"],
        "Median age (IQR)": [f"{child_df['age'].quantile(0.5)} ({child_df['age'].quantile(0.75) - child_df['age'].quantile(0.25)})"],
        "Sex": [struct_to_str(child_df["sexAtBirth"].value_counts().sort(by="sexAtBirth").to_struct(), "sexAtBirth")],
        "PH Diagnosis": [struct_to_str(child_df["diagnosisClassification (cleaned)"].value_counts().sort(by="diagnosisClassification (cleaned)").to_struct(), "diagnosisClassification (cleaned)")],
    })
)


In [ ]:
child_df.group_by("diagnosisClassification (cleaned)").agg(
    pl.format("{} ({}%)", pl.len(), (pl.len()/num_child_exc_dropouts * 100).round(1)).alias("Number of responders"),
    pl.format("{} ({})", pl.col("age").mean().round(1), pl.col("age").std().round(1)).alias("Mean age (std.)"),
    pl.format("{} ({})", pl.col("age").quantile(0.5), (pl.col("age").quantile(0.75) - pl.col("age").quantile(0.25))).alias("Median age (IQR)"),
    pl.col("sexAtBirth").value_counts().map_elements(lambda x: struct_to_str(x, "sexAtBirth")).alias("Sex"),
).sort(by="diagnosisClassification (cleaned)")

## Country breakdown

Paper p3 and Supp. Table S2. Responses by country.

In [ ]:
pl.Config.set_tbl_rows(100)
df.filter(
    pl.col("diagnosisClassification (cleaned)").is_not_null(),
).select(pl.col("country").value_counts()).unnest("country").sort(by="count", descending=True)

## Proxy vs Patient breakdown

Paper Figure 1 flow diagram. Adult patients vs proxies (caregiver, spouse, parent, other).

In [ ]:
print("Proxies:")
print(
    df.filter(pl.col("Are you answering this questionnaire as a... (cleaned)") != "Patient")
    .select(pl.col("sexAtBirth").value_counts())
    .with_columns(
        pl.col("sexAtBirth").struct.field("sexAtBirth").alias("sexAtBirth"),
        pl.col("sexAtBirth").struct.field("count").alias("count"),
        (pl.col("sexAtBirth").struct.field("count") / pl.col("sexAtBirth").struct.field("count").sum() * 100).round(1).alias("Percentage"),
    ).sort(by="sexAtBirth", descending=True)
)
print()
print("Patients:")
print(
    df.filter(pl.col("Are you answering this questionnaire as a... (cleaned)") == "Patient")
    .select(pl.col("sexAtBirth").value_counts())
    .with_columns(
        pl.col("sexAtBirth").struct.field("sexAtBirth").alias("sexAtBirth"),
        pl.col("sexAtBirth").struct.field("count").alias("count"),
        (pl.col("sexAtBirth").struct.field("count") / pl.col("sexAtBirth").struct.field("count").sum() * 100).round(1).alias("Percentage"),
    ).sort(by="sexAtBirth", descending=True)
)

# 2. Survey completion

Paper p5. Completion rates by section, region, and language.

## Completion by region

In [ ]:
adult_df.select(
    pl.col("region"),
    pl.col("partOfPHA").is_not_null().alias("Completed"),
).group_by("region").agg(
    pl.col("Completed").value_counts().map_elements(lambda x: struct_to_str(x, "Completed")),
).sort("region")

## Completion by language

In [ ]:
adult_df.select(
    pl.col("language"),
    pl.col("partOfPHA").is_not_null().alias("Completed"),
).group_by("language").agg(
    pl.col("Completed").value_counts().map_elements(lambda x: struct_to_str(x, "Completed")),
    pl.col("Completed").count().alias("Count"),
).sort("Count").reverse().select(
    pl.col("language"),
    pl.col("Completed"),
)

## Dropout over survey questions

Paper Supp. Figure S2. Proportion completing each survey checkpoint.

In [ ]:
mandatory_questions = [
    "onAnyAntiCoagulants",
    "experiencedSideEffects",
    "useOxygen",
    "hadTransplant",
    "disabledStatus",
    "ownSmartphone",
    "sixMinuteWalkTest",
    "filledOutPRO",
    "careOfSpecialist",
    "gettingDressedTakingAShower",
    "lowSelfEsteem",
    "diet",
    "partOfPHA",
]

prop_completed = adult_df.select(
    [((1 - (pl.col(col).null_count() / pl.col(col).count())) * 100).round(1).alias(col) for col in mandatory_questions]
).to_numpy()[0]

print(prop_completed)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=mandatory_questions,
    y=prop_completed,
    mode="lines+markers",
    name="Proportion completed",
))
fig.update_layout(
    title="Proportion of mandatory questions completed",
    xaxis_title="Question checkpoint",
    yaxis_title="Proportion completed",
)
fig.show()



# 3. Experiences of PH healthcare

Paper p5–7 and Table 2. Specialist care, healthcare costs, travel, and visit patterns by region and PH group.

## Table 2: Care under PH specialist centre

Paper Table 2, top section. Care under specialist centre by region and PH group.

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist").is_not_null()
)["careOfSpecialist"].value_counts().with_columns(
    pl.col("careOfSpecialist"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist").is_not_null()
).group_by(
    pl.col("region")
).agg(
    pl.col("careOfSpecialist").value_counts().map_elements(lambda x: struct_to_str(x, "careOfSpecialist"))
).sort(by="region")

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist").is_not_null()
).group_by(
    pl.col("diagnosisClassification (cleaned)")
).agg(
    pl.col("careOfSpecialist").value_counts().map_elements(lambda x: struct_to_str(x, "careOfSpecialist"))
).sort(by="diagnosisClassification (cleaned)")

## Time under care of specialist

Paper p5. Duration under specialist care (<1 year, 1–5 years, >5 years).

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("timeUnderCareOfSpecialist").is_not_null(),
)["timeUnderCareOfSpecialist"].value_counts().with_columns(
    pl.col("timeUnderCareOfSpecialist"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## Table 2: Healthcare cost coverage

Paper Table 2, bottom section. Financial coverage by region and PH group.

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("costCoveredBy").is_not_null(),
)["costCoveredBy"].value_counts().with_columns(
    pl.col("costCoveredBy"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("costCoveredBy").is_not_null(),
).select(
    pl.col("region"),
    pl.col("costCoveredBy")
).group_by("region").agg(
    pl.col("costCoveredBy").value_counts().map_elements(lambda x: struct_to_str(x, "costCoveredBy"))
).sort("region")

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("costCoveredBy").is_not_null(),
).select(
    pl.col("diagnosisClassification (cleaned)"),
    pl.col("costCoveredBy")
).group_by("diagnosisClassification (cleaned)").agg(
    pl.col("costCoveredBy").value_counts().map_elements(lambda x: struct_to_str(x, "costCoveredBy"))
).sort("diagnosisClassification (cleaned)")

## Rehabilitation

Paper p7. Access to rehabilitation services.

In [ ]:
adult_df.filter(
    pl.col("rehabilitationProvided").is_not_null(),
)["rehabilitationProvided"].value_counts().with_columns(
    pl.col("rehabilitationProvided"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("rehabilitationProvided") == "Yes",
    pl.col("rehabilitationCostCoveredBy").is_not_null(),
)["rehabilitationCostCoveredBy"].value_counts().with_columns(
    pl.col("rehabilitationCostCoveredBy"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## Travel to specialist centre

Paper p6. Mode of transport, distance (km), and travel time to PH centre.

In [ ]:
print("n=",
    adult_df.filter(
        pl.col("careOfSpecialist") == "Yes",
        pl.col("modeOfTransport").is_not_null().or_(pl.col("modeOfTransportOther").is_not_null()),
    )["modeOfTransport"].len()
)

df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("careOfSpecialist") == "Yes",
    pl.col("modeOfTransport").is_not_null().or_(pl.col("modeOfTransportOther").is_not_null()),
)["modeOfTransport"].str.split(";").list.explode().value_counts().with_columns(
    pl.col("modeOfTransport"),
    pl.col("count"),
    (pl.col("count") / 2225 * 100).round(1).alias("Percentage"),
)

In [ ]:
len(adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("modeOfTransportOther").is_not_null()
).select(
    pl.col("modeOfTransportOther"),
))

In [ ]:
from resources.replacements import DISTANCE_UNITS

transport = df.select(
    pl.col("region"),
    pl.col("distanceFromSpecialistNumber").cast(pl.Float64, strict=False),
    pl.col("distanceFromSpecialistUnit").replace(DISTANCE_UNITS),
    pl.col("lengthOfTravel")
).filter(
    ~pl.all_horizontal(pl.col("*").is_null()),
    pl.col("distanceFromSpecialistNumber").cast(pl.Float64, strict=False),
    (pl.col("distanceFromSpecialistUnit") == "km").or_(pl.col("distanceFromSpecialistUnit") == "miles"),
).with_columns(
    pl
    .when(pl.col("distanceFromSpecialistUnit").eq("miles"))
    .then(pl.col("distanceFromSpecialistNumber").cast(pl.Float64, strict=False) * 1.609344)
    .otherwise(pl.col("distanceFromSpecialistNumber"))
).with_columns(
    pl
    .when(pl.col("distanceFromSpecialistUnit").eq("miles"))
    .then(pl.col("distanceFromSpecialistUnit").str.replace("miles", "km"))
    .otherwise(pl.col("distanceFromSpecialistUnit"))
)

print(f"{round(transport["distanceFromSpecialistNumber"].mean(), 1)}km ({round(transport["distanceFromSpecialistNumber"].std(), 1)})")
print(f"{round(transport["distanceFromSpecialistNumber"].median(), 1)}km ({round(transport["distanceFromSpecialistNumber"].quantile(0.75) - transport["distanceFromSpecialistNumber"].quantile(0.25), 1)})")


In [ ]:
transport.group_by("region").agg(
    pl.col("distanceFromSpecialistNumber").mean().alias("Dist KM (mean)"),
    pl.col("distanceFromSpecialistNumber").std().alias("Dist KM (std)"),
    pl.col("distanceFromSpecialistNumber").median().alias("Dist KM (median)"),
    (pl.col("distanceFromSpecialistNumber").quantile(0.75) - pl.col("distanceFromSpecialistNumber").quantile(0.25)).alias("Dist KM (IQR)"),
).sort("region")

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
).select(
    pl.col("distanceFromSpecialistNumber"),
    pl.col("distanceFromSpecialistUnit"),
)

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("lengthOfTravel").is_not_null(),
)["lengthOfTravel"].value_counts().with_columns(
    pl.col("lengthOfTravel"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## Visit frequency, length, and accompaniment

Paper p7. How often patients visit, appointment duration, and who accompanies them.

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("frequencyOfSpecialist").is_not_null(),
)["frequencyOfSpecialist"].value_counts().with_columns(
    pl.col("frequencyOfSpecialist"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("lengthOfSpecialistVisit").is_not_null(),
)["lengthOfSpecialistVisit"].value_counts().with_columns(
    pl.col("lengthOfSpecialistVisit"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("visitAccompaniment").is_not_null(),
)["visitAccompaniment"].value_counts().with_columns(
    pl.col("visitAccompaniment"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

# 4. Perceptions of PH healthcare

Paper p7. Patient perceptions of appointments, remote healthcare, communication, and information quality.

## Difficulty or stressfulness of attending

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("difficultyOfVisit").is_not_null(),
)["difficultyOfVisit"].value_counts().with_columns(
    pl.col("difficultyOfVisit"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## In-person vs remote appointments

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("remoteAppointments").is_not_null(),
)["remoteAppointments"].value_counts().with_columns(
    pl.col("remoteAppointments"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## Remote healthcare preference

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("remotePreference").is_not_null(),
)["remotePreference"].value_counts().with_columns(
    pl.col("remotePreference"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## Quality of communication

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("qualityOfCommunication").is_not_null(),
)["qualityOfCommunication"].value_counts().with_columns(
    pl.col("qualityOfCommunication"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## Understanding of information provided

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("understandInformation").is_not_null(),
)["understandInformation"].value_counts().with_columns(
    pl.col("understandInformation"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## Checkups reflect experience

Paper p7. Whether patients feel consultations reflect how they actually feel.

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("checkupsReflectExperience").is_not_null(),
)["checkupsReflectExperience"].value_counts().with_columns(
    pl.col("checkupsReflectExperience"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

# 5. HRQoL and PROMs

Paper p7, Table 3, and Figures 3a–3b. Impact on wellbeing, daily activities, employment, side effects, symptom change, and PROM usage.

## Figure 3a: Self-reported emotions, cognition and wellbeing

Paper Figure 3a. 10 dimensions: sleep, fidgety/restless, angry, concentration, misunderstood, diet, self-esteem, isolated, fearful, guilt/embarrassment/hopelessness.

In [ ]:
qol_metrics = adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
).select(
    pl.col("diet").value_counts().sort(),
    pl.col("sleep").value_counts().sort(),
    pl.col("concentration").value_counts().sort(),
    pl.col("figetyRestressStressed").value_counts().sort(),
    pl.col("angryFrustrated").value_counts().sort(),
    pl.col("fearfulFrightened").value_counts().sort(),
    pl.col("lowSelfEsteem").value_counts().sort(),
    pl.col("guiltEmbarrassmentHoplessness").value_counts().sort(),
    pl.col("misunderstood").value_counts().sort(),
    pl.col("isolated").value_counts().sort(),
)

col_rename = {
    "guiltEmbarrassmentHoplessness": "Guilt, embarrassment, hoplessness",
    "fearfulFrightened": "Fearful/frightened",
    "isolated": "Isolated/no desire to socialise",
    "lowSelfEsteem": "Low self-esteem/confidence",
    "diet": "Diet",
    "misunderstood": "Misunderstood",
    "concentration": "Concentration",
    "angryFrustrated": "Angry/frustrated",
    "figetyRestressStressed": "Figety/restress/stressed",
    "sleep": "Sleep disorders/difficulties",
}

col_counts = {}
for col in qol_metrics.columns:
    col_counts[col] = {row[col]: row["count"] for row in qol_metrics.select(pl.col(col)).unnest(col).to_dicts()}

col_counts_normalised = {}
for col, value in col_counts.items():
    value = {k: v for k, v in value.items() if k is not None}
    col_counts_normalised[col] = {k: round(v / sum(value.values()) * 100, 1) for k, v in value.items()}

fig = make_subplots(rows=1, cols=1, horizontal_spacing=0.04)
fig = go.Figure(data=[
    go.Bar(
        y=[f"{col_rename[col]} (n={sum([v for k, v in col_counts[col].items() if k is not None])})" for col in col_counts_normalised.keys()],
        x=[_[response] for _ in col_counts_normalised.values()],
        orientation="h",
        name=response,
    ) for response in ["Never", "Rarely", "Sometimes", "Often", "Very often"]],
)
# Change the bar mode
fig.update_layout(
    title="Quality of Life symptoms",
    barmode='stack',
)

# fig.update_traces(marker_color="rgb(164, 0, 114)", row=1, col=11)
fig.update_xaxes(title_text="Percentage of respondents", title_font_size=12, range=[0, 100], ticksuffix="%")
fig.show()

## Figure 3b: Impact on activities of daily living

Paper Figure 3b. 5 dimensions: walking/stairs, leisure, domestic work, sexual relationships, getting dressed.

In [ ]:
# QoL function breakdown (Paper Figure 4)
temp = adult_df.select(
    pl.col("gettingDressedTakingAShower").filter(pl.col("gettingDressedTakingAShower").is_not_null()).value_counts().sort(),
    pl.col("sexualRealtionships").filter(pl.col("sexualRealtionships").is_not_null()).value_counts().sort(),
    pl.col("domesticWorkChores").filter(pl.col("domesticWorkChores").is_not_null()).value_counts().sort(),
    pl.col("leisureTimeActivities").filter(pl.col("leisureTimeActivities").is_not_null()).value_counts().sort(),
    pl.col("walkingShortDistanceStairs").filter(pl.col("walkingShortDistanceStairs") != "Not applicable (not mobile)").value_counts().sort(),
)

col_counts = {}
for col in temp.columns:
    col_counts[col] = {row[col]: row["count"] for row in temp.select(pl.col(col)).unnest(col).to_dicts()}

pd.DataFrame(col_counts)


In [ ]:
qol_function = adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
).select(
    pl.col("gettingDressedTakingAShower").filter(pl.col("gettingDressedTakingAShower").is_not_null()).value_counts().sort(),
    pl.col("sexualRealtionships").filter(pl.col("sexualRealtionships").is_not_null()).value_counts().sort(),
    pl.col("domesticWorkChores").filter(pl.col("domesticWorkChores").is_not_null()).value_counts().sort(),
    pl.col("leisureTimeActivities").filter(pl.col("leisureTimeActivities").is_not_null()).value_counts().sort(),
    pl.col("walkingShortDistanceStairs").filter(pl.col("walkingShortDistanceStairs") != "Not applicable (not mobile)").value_counts().sort(),
)

col_rename = {
    "gettingDressedTakingAShower": "Getting dressed/taking a shower",
    "sexualRealtionships": "Sexual relationships",
    "domesticWorkChores": "Domestic work/chores",
    "leisureTimeActivities": "Leisure time activities",
    "walkingShortDistanceStairs": "Walking/short distance/stairs",
}

col_counts = {}
for col in qol_function.columns:
    col_counts[col] = {row[col]: row["count"] for row in qol_function.select(pl.col(col)).unnest(col).to_dicts()}

col_counts_normalised = {}
for col, value in col_counts.items():
    value = {k: v for k, v in value.items() if k is not None}
    col_counts_normalised[col] = {k: round(v / sum(value.values()) * 100, 1) for k, v in value.items()}

fig = make_subplots(rows=1, cols=1, horizontal_spacing=0.04)
fig = go.Figure(data=[
    go.Bar(
        y=[f"{col_rename[col]} (n={sum([v for k, v in col_counts[col].items() if k is not None])})" for col in col_counts_normalised.keys()],
        x=[_[response] for _ in col_counts_normalised.values()],
        orientation="h",
        name=response,
    ) for response in ["Never", "Rarely", "Sometimes", "Often", "Very often"]],
)
# Change the bar mode
fig.update_layout(
    title="Quality of Life function",
    barmode='stack',
)

# fig.update_traces(marker_color="rgb(164, 0, 114)", row=1, col=11)
fig.update_xaxes(title_text="Percentage of respondents", title_font_size=12, range=[0, 100], ticksuffix="%")
fig.show()

## Activity level

Paper p7. Self-rated physical activity level.

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("selfRatedActivity").is_not_null(),
)["selfRatedActivity"].value_counts().select(
    pl.col("selfRatedActivity"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## Employment

Paper p7. Impact of PH on ability to work or study.

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
)["walkingShortDistanceStairs"].value_counts().filter(pl.col("walkingShortDistanceStairs") == "Not applicable (not mobile)")

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
).filter(
    pl.col("abilityToWorkAffected").is_not_null()
)["abilityToWorkAffected"].value_counts().sort(by="abilityToWorkAffected").with_columns(
    pl.col("abilityToWorkAffected"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## Side effects

Paper p7 and Supp. Table S7. Side effects from PH medications and discussion with provider.

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("experiencedSideEffects") == "Yes",
).select(
    pl.format(
        "{} ({}%)",
        stomach_nausea := pl.col("sideEffects").str.contains("Stomach upset or nausea").sum(),
        (stomach_nausea/pl.len() * 100).round(1)
    ).alias("Stomach upset or nausea"),

    pl.format(
        "{} ({}%)",
        heartburn := pl.col("sideEffects").str.contains("Heartburn").sum(),
        (heartburn/pl.len() * 100).round(1)
    ).alias("Heartburn"),

    pl.format(
        "{} ({}%)",
        nosebleeds := pl.col("sideEffects").str.contains("Nosebleeds").sum(),
        (nosebleeds/pl.len() * 100).round(1)
    ).alias("Nosebleeds"),

    pl.format(
        "{} ({}%)",
        skin_flushing := pl.col("sideEffects").str.contains("Skin flushing").sum(),
        (skin_flushing /pl.len() * 100).round(1)
    ).alias("Skin flushing"),

    pl.format(
        "{} ({}%)",
        muscle_aches := pl.col("sideEffects").str.contains("Muscle aches").sum(),
        (muscle_aches / pl.len() * 100).round(1)
    ).alias("Muscle aches"),

    pl.format(
        "{} ({}%)",
        sleeping := pl.col("sideEffects").str.contains("Trouble sleeping").sum(),
        (sleeping/pl.len() * 100).round(1)
    ).alias("Trouble sleeping"),

    pl.format(
        "{} ({}%)",
        sob := pl.col("sideEffects").str.contains("Shortness of breath").sum(),
        (sob/pl.len() * 100).round(1)
    ).alias("Shortness of breath"),

    pl.format(
        "{} ({}%)",
        nasal_congestion := pl.col("sideEffects").str.contains("Nasal congestion").sum(),
        (nasal_congestion/pl.len() * 100).round(1)
    ).alias("Nasal congestion"),

    pl.format(
        "{} ({}%)",
        other := (pl.col("sideEffectsOther").is_not_null()).sum(),
        (other/pl.len() * 100).round(1)
    ).alias("Other")
)


In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("experiencedSideEffects") == "Yes",
    pl.col("changingMedicationsDueToSideEffects").is_not_null()
)["changingMedicationsDueToSideEffects"].value_counts().with_columns(
    pl.col("changingMedicationsDueToSideEffects"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## Table 3: Change in symptoms since diagnosis

Paper Table 3, top section. Likert scale 1–5 (1=significantly worse, 5=fully recovered) by region and PH group.

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("changeSinceDiagnosis").is_not_null(),
)["changeSinceDiagnosis"].cast(pl.Int32).std()

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("changeSinceDiagnosis").is_not_null(),
).group_by(
    pl.col("region")
).agg(
    pl.col("changeSinceDiagnosis").cast(pl.Int32).count().alias("Count"),
    pl.col("changeSinceDiagnosis").cast(pl.Int32).mean().round(1).alias("Mean"),
    pl.col("changeSinceDiagnosis").cast(pl.Int32).std().round(1).alias("Standard Deviation"),
).sort(by="region")

In [ ]:
adult_df.filter(
    pl.col("careOfSpecialist") == "Yes",
    pl.col("changeSinceDiagnosis").is_not_null(),
).group_by(
    pl.col("diagnosisClassification (cleaned)")
).agg(
    pl.col("changeSinceDiagnosis").cast(pl.Int32).count().alias("Count"),
    pl.col("changeSinceDiagnosis").cast(pl.Int32).mean().round(1).alias("Mean"),
    pl.col("changeSinceDiagnosis").cast(pl.Int32).std().round(1).alias("Standard Deviation"),
).sort(by="diagnosisClassification (cleaned)")

## Table 3: Filled PROM questionnaire

Paper Table 3, middle section. Whether patients have ever completed a PROM, by region and PH group.

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("filledOutPRO").is_not_null(),
)["filledOutPRO"].value_counts().with_columns(
    pl.col("filledOutPRO"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("filledOutPRO").is_not_null(),
).group_by(
    pl.col("region")
).agg(
    pl.col("filledOutPRO").value_counts().map_elements(lambda x: struct_to_str(x, "filledOutPRO"))
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("filledOutPRO").is_not_null(),
).group_by(
    pl.col("diagnosisClassification (cleaned)")
).agg(
    pl.col("filledOutPRO").value_counts().map_elements(lambda x: struct_to_str(x, "filledOutPRO"))
).sort(by="diagnosisClassification (cleaned)")

## PROMs: which instruments, frequency, and method

Paper p7. PROM instruments used (CAMPHOR, emPHasis-10, LPHQ, PAH-SYMPACT), completion method and frequency.

In [ ]:
pros_df = adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("filledOutPRO") == "Yes",
    pl.col("whichPRO").is_not_null(),
)

print(pros_df["whichPRO"].value_counts().sum()["count"][0])
pros_df["whichPRO"].str.split(";").list.explode().value_counts()
pros_df.filter(~pl.col("whichPRO").str.contains(";"))["whichPRO"].value_counts()

In [ ]:
pros_df.with_columns(
    pl.when(pl.col("whichPRO").is_in(["EMPHASIS-10", "CAMPHOR", "LPHQ", "PAH-SYMPACT"]))
      .then(pl.col("whichPRO"))
      .otherwise(pl.lit("Other/Not sure")).alias("Grouped PRO")
).group_by(
    pl.col("Grouped PRO")
).agg(
    pl.col("frequencyOfPRO").filter(pl.col("frequencyOfPRO").is_not_null()).str.split(";").list.explode().value_counts().sort().map_elements(lambda x: struct_to_str(x, "frequencyOfPRO")),
    pl.col("methodOfCompletePRO").filter(pl.col("methodOfCompletePRO").is_not_null()).str.split(";").list.explode().value_counts().sort().map_elements(lambda x: struct_to_str(x, "methodOfCompletePRO")),
    pl.col("timeToCompletePRO").filter(pl.col("timeToCompletePRO").is_not_null()).value_counts().sort().map_elements(lambda x: struct_to_str(x, "timeToCompletePRO")),
    pl.col("accuratelyReflectedByPRO").filter(pl.col("accuratelyReflectedByPRO").is_not_null()).value_counts().sort().map_elements(lambda x: struct_to_str(x, "accuratelyReflectedByPRO")),
).sort(by="Grouped PRO").vstack(
    pl.DataFrame({
        "Grouped PRO": ["Total"],
        "frequencyOfPRO": [f"{pros_df.filter(pl.col('frequencyOfPRO').is_not_null())['frequencyOfPRO'].str.split(';').list.explode().value_counts().with_columns(pl.col('frequencyOfPRO'), pl.col('count'), (pl.col('count') / pl.col('count').sum() * 100).round(1).alias('percentage'))}"],
        "methodOfCompletePRO": [f"{pros_df.filter(pl.col('methodOfCompletePRO').is_not_null())['methodOfCompletePRO'].str.split(';').list.explode().value_counts().with_columns(pl.col('methodOfCompletePRO'), pl.col('count'), (pl.col('count')/pl.col('count').sum() * 100).round(1).alias('percentage'))}"],
        "timeToCompletePRO": [f"{pros_df.filter(pl.col('timeToCompletePRO').is_not_null())['timeToCompletePRO'].value_counts().with_columns(pl.col('timeToCompletePRO'), pl.col('count'), (pl.col('count') / pl.col('count').sum() * 100).round(1).alias('percentage'))}"],
        "accuratelyReflectedByPRO": [f"{pros_df.filter(pl.col('accuratelyReflectedByPRO').is_not_null())['accuratelyReflectedByPRO'].value_counts().with_columns(pl.col('accuratelyReflectedByPRO'), pl.col('count'), (pl.col('count') / pl.col('count').sum() * 100).round(1).alias('percentage'))}"],
    })
)



In [ ]:
test = "PAH-SYMPACT"
num_responses_for_test = len(pros_df.filter(pl.col("whichPRO").str.contains(test)))

print(f"How often do you fill out the PRO questonnaire? Test: {test}\n")

print("Only once: ", num_w_percentage_float(pros_df.filter(
    pl.col("whichPRO") == test
)["frequencyOfPRO"].str.split(";").list.explode().str.contains("Only once").sum(), num_responses_for_test))

print("Every time I have an appointment: ", num_w_percentage_float(pros_df.filter(
    pl.col("whichPRO") == test
)["frequencyOfPRO"].str.split(";").list.explode().str.contains("Every time").sum(), num_responses_for_test))

print("At regular intervals: ", num_w_percentage_float(pros_df.filter(
    pl.col("whichPRO") == test
)["frequencyOfPRO"].str.split(";").list.explode().str.contains("regular").sum(), num_responses_for_test))

print("In a clinical trial: ", num_w_percentage_float(pros_df.filter(
    pl.col("whichPRO") == test
)["frequencyOfPRO"].str.split(";").list.explode().str.contains("clinical trial").sum(), num_responses_for_test))

print("Other/unclear: ", num_w_percentage_float(pros_df["whichPRO"].str.contains(test).sum() - len(pros_df.filter(pl.col("whichPRO") == test)), num_responses_for_test))

### PROMs feedback and management alteration

Paper p7. Whether patients received feedback on PROMs and whether management was altered.

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("feedbackOnPRO").is_not_null(),
)["feedbackOnPRO"].value_counts().with_columns(
    pl.col("feedbackOnPRO"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("alterationBasedOnPRO").is_not_null(),
)["alterationBasedOnPRO"].value_counts().with_columns(
    pl.col("alterationBasedOnPRO"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## Pregnancy concerns

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("sexAtBirth") == "Female",
    pl.col("worryAboutPregnancy").is_not_null(),
)["worryAboutPregnancy"].value_counts().select(
    pl.col("worryAboutPregnancy"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## QoL by PH group

QoL metrics (emotions, daily activities) broken down by PH diagnosis group.

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
).group_by("diagnosisClassification (cleaned)").agg(
    pl.col("gettingDressedTakingAShower").value_counts().map_elements(lambda x: struct_to_str(x, "gettingDressedTakingAShower")),
    pl.col("walkingShortDistanceStairs").value_counts().map_elements(lambda x: struct_to_str(x, "walkingShortDistanceStairs")),
    pl.col("domesticWorkChores").value_counts().map_elements(lambda x: struct_to_str(x, "domesticWorkChores")),
    pl.col("leisureTimeActivities").value_counts().map_elements(lambda x: struct_to_str(x, "leisureTimeActivities")),
    pl.col("sexualRealtionships").value_counts().map_elements(lambda x: struct_to_str(x, "sexualRealtionships")),
    pl.col("lowSelfEsteem").value_counts().map_elements(lambda x: struct_to_str(x, "lowSelfEsteem")),
    pl.col("isolated").value_counts().map_elements(lambda x: struct_to_str(x, "isolated")),
    pl.col("fearfulFrightened").value_counts().map_elements(lambda x: struct_to_str(x, "fearfulFrightened")),
    pl.col("angryFrustrated").value_counts().map_elements(lambda x: struct_to_str(x, "angryFrustrated")),
    pl.col("misunderstood").value_counts().map_elements(lambda x: struct_to_str(x, "misunderstood")),
    pl.col("guiltEmbarrassmentHoplessness").value_counts().map_elements(lambda x: struct_to_str(x, "guiltEmbarrassmentHoplessness")),
    pl.col("diet").value_counts().map_elements(lambda x: struct_to_str(x, "diet")),
    pl.col("sleep").value_counts().map_elements(lambda x: struct_to_str(x, "sleep")),
    pl.col("concentration").value_counts().map_elements(lambda x: struct_to_str(x, "concentration")),
    pl.col("figetyRestressStressed").value_counts().map_elements(lambda x: struct_to_str(x, "figetyRestressStressed")),
).sort(by="diagnosisClassification (cleaned)")

# 6. Self-monitoring and digital health

Paper p7, Table 3, and Supp. Tables S9–S12. Self-monitoring practices, smartphone/wearable ownership, and data sharing by region and age.

## Table 3: Self-monitor

Paper Table 3, self-monitor row. Self-monitoring status by region and age.

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("currentlySelfMonitor").is_not_null(),
)["currentlySelfMonitor"].value_counts().with_columns(
    pl.col("currentlySelfMonitor"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("currentlySelfMonitor").is_not_null(),
).group_by(
    pl.col("region")
).agg(
    pl.col("currentlySelfMonitor").value_counts().map_elements(lambda x: struct_to_str(x, "currentlySelfMonitor"))
).sort(by="region")

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("currentlySelfMonitor").is_not_null(),
).group_by(
    (pl.col("age") < 20).alias("<20"),
    ((20 <= pl.col("age")) & (pl.col("age") < 30)).alias("20-29"),
    ((30 <= pl.col("age")) & (pl.col("age") < 40)).alias("30-39"),
    ((40 <= pl.col("age")) & (pl.col("age") < 50)).alias("40-49"),
    ((50 <= pl.col("age")) & (pl.col("age") < 60)).alias("50-59"),
    ((60 <= pl.col("age")) & (pl.col("age") < 70)).alias("60-69"),
    ((70 <= pl.col("age")) & (pl.col("age") < 80)).alias("70-79"),
    ((80 <= pl.col("age")) & (pl.col("age") < 90)).alias("80-89"),
    (pl.col("age") >= 90).alias(">=90"),
).agg(
    pl.col("currentlySelfMonitor").value_counts().map_elements(lambda x: struct_to_str(x, "currentlySelfMonitor"))
)

## How do you monitor

Monitoring method (paper/calendar, smartphone app, specialised PH app) by region and age.

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("howDoYouMonitor").is_not_null(),
).group_by(
    pl.col("region")
).agg(
    pl.col("howDoYouMonitor").value_counts().map_elements(lambda x: struct_to_str(x, "howDoYouMonitor"))
).sort(by="region")

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("howDoYouMonitor").is_not_null(),
).group_by(
    (pl.col("age") < 20).alias("<20"),
    ((20 <= pl.col("age")) & (pl.col("age") < 30)).alias("20-29"),
    ((30 <= pl.col("age")) & (pl.col("age") < 40)).alias("30-39"),
    ((40 <= pl.col("age")) & (pl.col("age") < 50)).alias("40-49"),
    ((50 <= pl.col("age")) & (pl.col("age") < 60)).alias("50-59"),
    ((60 <= pl.col("age")) & (pl.col("age") < 70)).alias("60-69"),
    ((70 <= pl.col("age")) & (pl.col("age") < 80)).alias("70-79"),
    ((80 <= pl.col("age")) & (pl.col("age") < 90)).alias("80-89"),
    (pl.col("age") >= 90).alias(">=90"),
).agg(
    pl.col("howDoYouMonitor").value_counts().map_elements(lambda x: struct_to_str(x, "howDoYouMonitor"))
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("howDoYouMonitor").is_not_null(),
)["howDoYouMonitor"].value_counts().with_columns(
    pl.col("howDoYouMonitor"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## Share self-collected information

Paper p7. Whether patients share self-collected health data with their clinical team.

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("shareCollectedInformation").is_not_null(),
)["shareCollectedInformation"].value_counts().with_columns(
    pl.col("shareCollectedInformation"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("shareCollectedInformation").is_not_null(),
).group_by("region").agg(
    pl.col("shareCollectedInformation").value_counts().map_elements(lambda x: struct_to_str(x, "shareCollectedInformation"))
).sort(by="region")

## Smartphone ownership

Paper p7 and Supp. Tables S10–S11. Smartphone ownership by region and age.

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("ownSmartphone").is_not_null(),
)["ownSmartphone"].value_counts().with_columns(
    pl.col("ownSmartphone"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("ownSmartphone").is_not_null(),
).group_by("region").agg(
    pl.col("ownSmartphone").value_counts().map_elements(lambda x: struct_to_str(x, "ownSmartphone"))
).sort(by="region")

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("ownSmartphone").is_not_null(),
).group_by(
    (pl.col("age") < 20).alias("<20"),
    ((20 <= pl.col("age")) & (pl.col("age") < 30)).alias("20-29"),
    ((30 <= pl.col("age")) & (pl.col("age") < 40)).alias("30-39"),
    ((40 <= pl.col("age")) & (pl.col("age") < 50)).alias("40-49"),
    ((50 <= pl.col("age")) & (pl.col("age") < 60)).alias("50-59"),
    ((60 <= pl.col("age")) & (pl.col("age") < 70)).alias("60-69"),
    ((70 <= pl.col("age")) & (pl.col("age") < 80)).alias("70-79"),
    ((80 <= pl.col("age")) & (pl.col("age") < 90)).alias("80-89"),
    (pl.col("age") >= 90).alias(">=90"),
).agg(
    pl.col("ownSmartphone").value_counts().map_elements(lambda x: struct_to_str(x, "ownSmartphone"))
)

## Wearable / smartwatch ownership

Paper p7 and Supp. Table S12. Activity tracking device usage by region and age.

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("ownWearable").is_not_null(),
)["ownWearable"].value_counts().with_columns(
    pl.col("ownWearable"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("ownWearable").is_not_null(),
).group_by("region").agg(
    pl.col("ownWearable").value_counts().map_elements(lambda x: struct_to_str(x, "ownWearable"))
).sort(by="region")

In [ ]:
adult_df.filter(
    pl.col("ownWearable").is_not_null(),
).group_by(
    (pl.col("age") < 20).alias("<20"),
    ((20 <= pl.col("age")) & (pl.col("age") < 30)).alias("20-29"),
    ((30 <= pl.col("age")) & (pl.col("age") < 40)).alias("30-39"),
    ((40 <= pl.col("age")) & (pl.col("age") < 50)).alias("40-49"),
    ((50 <= pl.col("age")) & (pl.col("age") < 60)).alias("50-59"),
    ((60 <= pl.col("age")) & (pl.col("age") < 70)).alias("60-69"),
    ((70 <= pl.col("age")) & (pl.col("age") < 80)).alias("70-79"),
    ((80 <= pl.col("age")) & (pl.col("age") < 90)).alias("80-89"),
    (pl.col("age") >= 90).alias(">=90"),
).agg(
    pl.col("ownWearable").value_counts().map_elements(lambda x: struct_to_str(x, "ownWearable"))
)

## Confidence in using apps to monitor health

Confidence in using health apps, split by smartphone ownership, region, and age.

In [ ]:
print("Own smartphone: ")

adult_df.filter(
    pl.col("ownSmartphone").str.contains("Yes"),
)["confidenceInUsingAppsToMonitor"].value_counts().with_columns(
    pl.col("confidenceInUsingAppsToMonitor"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
print("Do not own a smart phone:")

adult_df.filter(
    ~pl.col("ownSmartphone").str.contains("Yes"),
)["confidenceInUsingAppsToMonitor"].value_counts().with_columns(
    pl.col("confidenceInUsingAppsToMonitor"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("ownSmartphone").is_not_null(),
).group_by("region").agg(
    pl.col("confidenceInUsingAppsToMonitor").value_counts().map_elements(lambda x: struct_to_str(x, "confidenceInUsingAppsToMonitor"))
).sort("region")

In [ ]:
adult_df.filter(
    pl.col("ownSmartphone").is_not_null(),
).group_by(
    (pl.col("age") < 20).alias("<20"),
    ((20 <= pl.col("age")) & (pl.col("age") < 30)).alias("20-29"),
    ((30 <= pl.col("age")) & (pl.col("age") < 40)).alias("30-39"),
    ((40 <= pl.col("age")) & (pl.col("age") < 50)).alias("40-49"),
    ((50 <= pl.col("age")) & (pl.col("age") < 60)).alias("50-59"),
    ((60 <= pl.col("age")) & (pl.col("age") < 70)).alias("60-69"),
    ((70 <= pl.col("age")) & (pl.col("age") < 80)).alias("70-79"),
    ((80 <= pl.col("age")) & (pl.col("age") < 90)).alias("80-89"),
    (pl.col("age") >= 90).alias(">=90"),
).agg(
    pl.col("confidenceInUsingAppsToMonitor").value_counts().map_elements(lambda x: struct_to_str(x, "confidenceInUsingAppsToMonitor"))
)

## 6-minute walk test

Whether patients have had a 6-minute walk test, by region and age.

In [ ]:
adult_df.filter(
    pl.col("sixMinuteWalkTest").is_not_null(),
)["sixMinuteWalkTest"].value_counts().with_columns(
    pl.col("sixMinuteWalkTest"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("sixMinuteWalkTest").is_not_null(),
).group_by("region").agg(
    pl.col("sixMinuteWalkTest").value_counts().map_elements(lambda x: struct_to_str(x, "sixMinuteWalkTest"))
).sort(by="region")

In [ ]:
adult_df.filter(
    pl.col("sixMinuteWalkTest").is_not_null(),
).group_by(
    (pl.col("age") < 20).alias("<20"),
    ((20 <= pl.col("age")) & (pl.col("age") < 30)).alias("20-29"),
    ((30 <= pl.col("age")) & (pl.col("age") < 40)).alias("30-39"),
    ((40 <= pl.col("age")) & (pl.col("age") < 50)).alias("40-49"),
    ((50 <= pl.col("age")) & (pl.col("age") < 60)).alias("50-59"),
    ((60 <= pl.col("age")) & (pl.col("age") < 70)).alias("60-69"),
    ((70 <= pl.col("age")) & (pl.col("age") < 80)).alias("70-79"),
    ((80 <= pl.col("age")) & (pl.col("age") < 90)).alias("80-89"),
    (pl.col("age") >= 90).alias(">=90"),
).agg(
    pl.col("sixMinuteWalkTest").value_counts().map_elements(lambda x: struct_to_str(x, "sixMinuteWalkTest"))
)

# 7. Disability and support

Paper p7–8, Table 3, and Supp. Tables S13–S14. Disability registration, benefits, rehabilitation, patient organisation membership, and support.

## Table 3: Registered as disabled

Paper Table 3, bottom row. Disability registration by region and PH group.

In [ ]:
df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("disabledStatus").is_not_null()
)["disabledStatus"].value_counts().with_columns(
    pl.col("disabledStatus"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("disabledStatus").is_not_null()
).select(
    pl.col("region"),
    pl.col("disabledStatus"),
).group_by("region").agg(
    pl.col("disabledStatus").value_counts().map_elements(lambda x: struct_to_str(x, "disabledStatus"))
).sort("region")

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("disabledStatus").is_not_null()
).select(
    pl.col("diagnosisClassification (cleaned)"),
    pl.col("disabledStatus"),
).group_by("diagnosisClassification (cleaned)").agg(
    pl.col("disabledStatus").value_counts().map_elements(lambda x: struct_to_str(x, "disabledStatus"))
).sort("diagnosisClassification (cleaned)")

## Disability benefits

Paper p7. Types of benefits received (parking, financial, enhanced care, reduced hours).

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("disabledStatus") == "Yes",
).select(
    pl.format(
        "{} ({}%)",
        special_care := pl.col("disabledStatusBenefits").str.contains("Special care").sum(),
        (special_care/pl.len() * 100).round(1)
    ).alias("Special care"),

    pl.format(
        "{} ({}%)",
        parking := pl.col("disabledStatusBenefits").str.contains("Parking").sum(),
        (parking/pl.len() * 100).round(1)
    ).alias("Parking"),

    pl.format(
        "{} ({}%)",
        financial := pl.col("disabledStatusBenefits").str.contains("Financial benefits").sum(),
        (financial/pl.len() * 100).round(1)
    ).alias("Financial benefits"),

    pl.format(
        "{} ({}%)",
        reduced_working_hours := pl.col("disabledStatusBenefits").str.contains("Reduced working hours").sum(),
        (reduced_working_hours/pl.len() * 100).round(1)
    ).alias("Reduced working hours"),

    pl.format(
        "{} ({}%)",
        other := (pl.col("disabledStatusBenefitsOther").str.len_chars() > 0).sum(),
        (other/pl.len() * 100).round(1)
    ).alias("Other"),
)

## Patient PH organisation membership and support

Paper p7–8 and Supp. Table S14. Membership rates and most valued support areas.

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("partOfPHA").is_not_null(),
)["partOfPHA"].value_counts().select(
    pl.col("partOfPHA"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("partOfPHA") == "Yes",
).select(
    pl.format(
        "{} ({}%)",
        information := pl.col("mainBenefitsOfPHA").str.contains("Information/education for patients and their friends/family").sum(),
        (information/pl.len() * 100).round(1)
    ).alias("Information/eduation for patients and their friends/family"),

    pl.format(
        "{} ({}%)",
        meetings := pl.col("mainBenefitsOfPHA").str.contains("Patient meetings/support groups").sum(),
        (meetings/pl.len() * 100).round(1)
    ).alias("Patient meetings/support groups"),

    pl.format(
        "{} ({}%)",
        helpline := pl.col("mainBenefitsOfPHA").str.contains("24/7 phone helpline").sum(),
        (helpline/pl.len() * 100).round(1)
    ).alias("24/7 phone helpline"),

    pl.format(
        "{} ({}%)",
        social_support := pl.col("mainBenefitsOfPHA").str.contains("Help with social support applications").sum(),
        (social_support/pl.len() * 100).round(1)
    ).alias("Help with social support applications"),

    pl.format(
        "{} ({}%)",
        travel_arrangements := pl.col("mainBenefitsOfPHA").str.contains("Help with travel arrangements").sum(),
        (travel_arrangements/pl.len() * 100).round(1)
    ).alias("Help with travel arrangements"),

    pl.format(
        "{} ({}%)",
        access_to_treatment := pl.col("mainBenefitsOfPHA").str.contains("Help with access to treatment").sum(),
        (access_to_treatment/pl.len() * 100).round(1)
    ).alias("Help with access to treatment"),

    pl.format(
        "{} ({}%)",
        devices := pl.col("mainBenefitsOfPHA").str.contains("Access to helpful devices").sum(),
        (devices/pl.len() * 100).round(1)
    ).alias("Access to helpful devices"),

    pl.format(
        "{} ({}%)",
        communication := pl.col("mainBenefitsOfPHA").str.contains("Help with communication between you and your healthcare provider").sum(),
        (communication/pl.len() * 100).round(1)
    ).alias("Help with communication between you and your healthcare provider"),

    pl.format(
        "{} ({}%)",
        website := pl.col("mainBenefitsOfPHA").str.contains("Access to an informative website").sum(),
        (website/pl.len() * 100).round(1)
    ).alias("Access to an informative website"),

    pl.format(
        "{} ({}%)",
        magazine := pl.col("mainBenefitsOfPHA").str.contains("Access to a regular magazine").sum(),
        (magazine/pl.len() * 100).round(1)
    ).alias("Access to a regular magazine"),

    pl.format(
        "{} ({}%)",
        awareness := pl.col("mainBenefitsOfPHA").str.contains("Awareness activities").sum(),
        (awareness/pl.len() * 100).round(1)
    ).alias("Awareness activities"),

    pl.format(
        "{} ({}%)",
        financial := pl.col("mainBenefitsOfPHA").str.contains("Financial support in emergencies").sum(),
        (financial/pl.len() * 100).round(1)
    ).alias("Financial support in emergencies"),

    pl.format(
        "{} ({}%)",
        other := (pl.col("mainBenefitsOfPHAOther").str.len_chars() > 0).sum(),
        (other/pl.len() * 100).round(1)
    ).alias("Other"),

)

## Who to discuss negative QoL impact with

Paper p7. Who patients discuss wellbeing impacts with (specialist, family, partner, etc.).

In [ ]:
print(adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("negativeImpactOnQoL").is_not_null(),
).select(
    pl.format(
        "{} ({}%)",
        ph_specialist := pl.col("negativeImpactOnQoL").str.contains("Pulmonary Hypertension").sum(),
        (ph_specialist/pl.len() * 100).round(1)
    ).alias("Pulmonary Hypertension (PH) Specialist"),

    pl.format(
        "{} ({}%)",
        other_specialist := pl.col("negativeImpactOnQoL").str.contains("Other Specialists").sum(),
        (other_specialist/pl.len() * 100).round(1)
    ).alias("Other Specialists"),

    pl.format(
        "{} ({}%)",
        family_doctor := pl.col("negativeImpactOnQoL").str.contains("Family doctor/GP").sum(),
        (family_doctor/pl.len() * 100).round(1)
    ).alias("Family doctor/GP"),

    pl.format(
        "{} ({}%)",
        counsellor := pl.col("negativeImpactOnQoL").str.contains("Counsellor/psychologist").sum(),
        (counsellor/pl.len() * 100).round(1)
    ).alias("Counsellor/psychologist"),

    pl.format(
        "{} ({}%)",
        nurse := pl.col("negativeImpactOnQoL").str.contains("Nurse").sum(),
        (nurse/pl.len() * 100).round(1)
    ).alias("Nurse"),

    pl.format(
        "{} ({}%)",
        patient_org := pl.col("negativeImpactOnQoL").str.contains("Patient Organisation").sum(),
        (patient_org/pl.len() * 100).round(1)
    ).alias("Patient Organisation"),

    pl.format(
        "{} ({}%)",
        significant_other := pl.col("negativeImpactOnQoL").str.contains("Significant Other/Life Partner").sum(),
        (significant_other/pl.len() * 100).round(1)
    ).alias("Significant Other/Life Partner"),

    pl.format(
        "{} ({}%)",
        family_friend := pl.col("negativeImpactOnQoL").str.contains("Family/friends").sum(),
        (family_friend/pl.len() * 100).round(1)
    ).alias("Family/friends"),

))

adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
).select(
    pl.format(
        "{} ({}%)",
        other := (pl.col("negativeImpactOnQoLOther").str.len_chars() > 0).sum(),
        (other/pl.len() * 100).round(1)
    ).alias("Other"),
)

# 8. Research participation

Paper p8–9, Figure 4, and Supp. Table S15. Clinical trial participation, willingness, experience, and data sharing.

## Research trial participation and experience

Paper Figure 4. Cascade: participation → experience → willingness to participate again, by region.

In [ ]:
grouping_col = "region"  # or "diagnosisClassification (cleaned)"
# df.filter(
#     pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
# ).group_by(grouping_col).agg(
#     pl.col("takenPartInResearchTrial").value_counts().map_elements(lambda x: struct_to_str(x, "takenPartInResearchTrial")),
# ).sort(grouping_col)


# df.filter(
#     pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
#     pl.col("takenPartInResearchTrial") == "No",
# ).group_by(grouping_col).agg(
#     pl.col("everConsideredTrial").value_counts().map_elements(lambda x: struct_to_str(x, "everConsideredTrial")),
# ).sort(grouping_col)


# df.filter(
#     pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
#     pl.col("takenPartInResearchTrial") == "No",
#     pl.col("everConsideredTrial") == "No, please specify",
# ).group_by(grouping_col).agg(
#     (pl.col("everConsideredTrialWhyNot").str.len_chars() > 1).sum(),
# ).sort(grouping_col)


# df.filter(
#     pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
#     pl.col("takenPartInResearchTrial") == "Yes",
# ).group_by(grouping_col).agg(
#     pl.col("researchTrialExperience").value_counts().map_elements(lambda x: struct_to_str(x, "researchTrialExperience")),
# ).sort("diagnosisClassification (cleaned)")


# df.filter(
#     pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
#     pl.col("takenPartInResearchTrial") == "Yes",
#     pl.col("researchTrialExperience").is_in(["Good", "Very good"])
# ).group_by(grouping_col).agg(
#     pl.col("wouldParticipateAgainInTrial").value_counts().map_elements(lambda x: struct_to_str(x, "wouldParticipateAgainInTrial")),
# ).sort(grouping_col)


adult_df.filter(
    pl.col("takenPartInResearchTrial") == "Yes",
    pl.col("researchTrialExperience").is_in(["Bad", "Very bad"])
).group_by(grouping_col).agg(
    pl.col("wouldParticipateAgainInTrial").value_counts().map_elements(lambda x: struct_to_str(x, "wouldParticipateAgainInTrial")),
).sort(grouping_col)

## Willingness to share anonymised data

Paper p9. Willingness to allow researchers to use anonymised healthcare data.

In [ ]:
adult_df.filter(
    pl.col("allowAnonymousUseOfData").is_not_null(),
)["allowAnonymousUseOfData"].value_counts().select(
    pl.col("allowAnonymousUseOfData"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
df.filter(
    pl.col("allowAnonymousUseOfData").is_in(["Yes", "No"]),
)["allowAnonymousUseOfData"].value_counts().with_columns(
    pl.col("allowAnonymousUseOfData"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

## Email for future research

Paper p9. Proportion who opted in to be contacted for future survey-based research.

In [ ]:
adult_df.filter(
    pl.col("allowAnonymousUseOfData").is_not_null(),
)["gaveEmail"].value_counts().with_columns(
    pl.col("gaveEmail"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
print(f"Email left for future surveys: {(df['gaveEmail'] == 'x').sum()} ({((df['gaveEmail'] == 'x').sum() / df.shape[0]) * 100:.1f}%)")

# 9. PH groups 1 and 4 specific results

Paper p9–10, Figure 5, and Supp. Tables S16–S18. Time to diagnosis, genetic testing (Group 1), and treatment patterns (Groups 1 and 4).

## Figure 5a: Time to diagnosis – Group 1 (iPAH and hPAH)

Paper Figure 5a. Time from symptom onset to diagnosis for idiopathic/heritable PAH, by region.

In [ ]:
adult_df.filter(
    pl.col("diagnosisClassification (cleaned)") == "Group 1 - pulmonary arterial hypertension (PAH)",
    pl.col("groupOneSubclassification (cleaned)").is_in(["Idiopathic pulmonary arterial hypertension (iPAH)", "Heritable/genetic pulmonary arterial hypertension (hPAH)"]),
    pl.col("timeToDiagnosis").is_not_null(),
).group_by(
    pl.col("region"),
).agg(
    (pl.col("timeToDiagnosis") == "1 - 6 months").sum() / pl.col("timeToDiagnosis").len() * 100,
).sort("region")


## Figure 5b: Time to diagnosis – Group 4 (CTEPH)

Paper Figure 5b. Time from symptom onset to diagnosis for CTEPH, by region.

## Genetic testing (Group 1 PAH)

Paper p9 and Supp. Table S17. Genetic testing rates, reasons for no testing, and results by region.

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("diagnosisClassification (cleaned)") == "Group 1 - pulmonary arterial hypertension (PAH)",
    pl.col("hadGeneticTesting").is_not_null()
)["hadGeneticTesting"].value_counts().with_columns(
    pl.col("hadGeneticTesting"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("diagnosisClassification (cleaned)") == "Group 1 - pulmonary arterial hypertension (PAH)",
    pl.col("hadGeneticTesting").is_not_null()
).group_by("region").agg(
    pl.col("hadGeneticTesting").value_counts().map_elements(lambda x: struct_to_str(x, "hadGeneticTesting"))
).sort(by="region")

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("diagnosisClassification (cleaned)") == "Group 1 - pulmonary arterial hypertension (PAH)",
    pl.col("hadGeneticTesting") == "No",
    pl.col("reasonForNoTesting").is_not_null(),
)["reasonForNoTesting"].value_counts().with_columns(
    pl.col("reasonForNoTesting"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("diagnosisClassification (cleaned)") == "Group 1 - pulmonary arterial hypertension (PAH)",
    pl.col("hadGeneticTesting") == "No",
    pl.col("reasonForNoTesting").is_not_null(),
).group_by(
    pl.col("region")
).agg(
    pl.col("reasonForNoTesting").value_counts().map_elements(lambda x: struct_to_str(x, "reasonForNoTesting"))
).sort(by="region")

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("diagnosisClassification (cleaned)") == "Group 1 - pulmonary arterial hypertension (PAH)",
    pl.col("hadGeneticTesting").str.contains("Yes")
)["resultOfTesting"].value_counts().with_columns(
    pl.col("resultOfTesting"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("diagnosisClassification (cleaned)") == "Group 1 - pulmonary arterial hypertension (PAH)",
    pl.col("hadGeneticTesting").str.contains("Yes")
).group_by(
    pl.col("region")
).agg(
    pl.col("resultOfTesting").value_counts().map_elements(lambda x: struct_to_str(x, "resultOfTesting"))
).sort(by="region")

## Supplementary Table S16: Group 1 PAH – Treatment

Medication therapy breakdown for Group 1 PAH. Drug definitions are in the Setup section above.

## Supplementary Table S18: Group 4 CTEPH – Treatment

PEA, BPA, anticoagulants, and vasodilator therapy for CTEPH patients by region.

In [ ]:
g4 = adult_df.filter(
    pl.col("diagnosisClassification (cleaned)") == "Group 4 - Chronic thromboembolic pulmonary hypertension (CTEPH)",
).with_columns(
    ((get_parenteral(pl.col("namesOfMedications")) + get_parenteral(pl.col("namesOfMedicationsOther"))) > 0).alias("takingParenteralDrugs"),
    (pl.col("namesOfMedicationsOther").str.to_lowercase().str.replace(", ", ";").str.replace(",", ";").str.split(";").list.set_intersection(unlicensed_drugs).list.len() > 0).alias("Unlicensed/trial drugs"),
    get_num_of_vasodilators(pl.col("namesOfMedications"), pl.col("namesOfMedicationsOther")).alias("Number of PH medical therapies (vasodilators)"),
)

g4.group_by("region").agg(
    # Respondents
    num_w_percentage_denom(pl.col("offeredPEA").len(), len(g4), "Number of individuals"),

    # PH medical therapy
    num_w_percentage_expr(pl.col("Number of PH medical therapies (vasodilators)") == 1, "Single"),
    num_w_percentage_expr(pl.col("Number of PH medical therapies (vasodilators)") == 2, "Dual"),
    num_w_percentage_expr(pl.col("Number of PH medical therapies (vasodilators)") == 3, "Triple"),


    # Taking parenteral drugs
    num_w_percentage_expr(pl.col("takingParenteralDrugs"), "Taking parenteral drugs"),

    # Unlicensed drugs
    pl.format("{} ({}%)", unlicensed := (pl.col("Unlicensed/trial drugs")).sum(), (unlicensed/pl.len() * 100).round(1)).alias("Unlicensed/trial drugs"),

    # Calcium channel blockers
    num_w_percentage_expr(
        pl.col("namesOfMedications").str.contains(
            r"Amlodipine \(Istin\)|Diltiazem \(Adizem, Angitil, Dilcardia, Retalzem, Slozem, Tildiem, Uard, Viazem, Zemtard\)|Nifedipine \(Adalat, Adanif, Adipine, Coracten, Dexipress, Fortipine, Neozipine, Nidef, Nifedipress, Tensipine, Valni\)|Nircadipine \(Cardene\)"
            ),
        "Calcium channel blockers"
    ),

    # Anticoagulants
    num_w_percentage_expr(pl.col("antiCoagulants").str.contains(r"Warfarin or Acenocoumarol tablets \- these require regular blood tests to check the dose"), "VKA"),
    num_w_percentage_expr(pl.col("antiCoagulants").str.contains(r"Dabigatran, rivaroxaban, apixaban, edoxaban or betrixaban tablets"), "DOAC"),
    num_w_percentage_expr(pl.col("antiCoagulants").str.contains(r"Injections under the skin \(e.g. heparin drugs\) - there are lots of types"), "LMWH"),

    # PEA
    num_w_percentage_expr((pl.col("hadPEA") == "Yes"), "Recieved PEA"),
    pl.format(
        "{} ({})",
        pl.col("PEARating").filter(pl.col("PEARating").is_not_null(), pl.col("PEARating").is_between(1, 5)).cast(pl.Int32).mean().round(1),
        pl.col("PEARating").filter(pl.col("PEARating").is_not_null(), pl.col("PEARating").is_between(1, 5)).cast(pl.Int32).std().round(1),
    ).alias("PEA Rating"),
    pl.col("PEARating").filter(pl.col("PEARating").is_not_null(), pl.col("PEARating").is_between(1, 5)).cast(pl.Int32).value_counts().map_elements(lambda x: struct_to_str(x, "PEARating")).alias("PEARatingBreakdown"),

    # BPA
    num_w_percentage_expr(pl.col("hadBPA").str.contains("Yes BPA"), "Had BPA"),
    pl.col("sessionsOfBPA").filter(pl.col("sessionsOfBPA").is_not_null(), pl.col("sessionsOfBPA").str.contains("[-More]")).value_counts().map_elements(lambda x: struct_to_str(x, "sessionsOfBPA")).alias("Sessions of BPA"),
    pl.format(
        "{} ({})",
        pl.col("BPARating").filter(pl.col("BPARating").is_not_null(), pl.col("BPARating").is_between(1, 5)).cast(pl.Int32).mean().round(1),
        pl.col("BPARating").filter(pl.col("BPARating").is_not_null(), pl.col("BPARating").is_between(1, 5)).cast(pl.Int32).std().round(1),
    ).alias("Mean BPA Rating"),
    pl.col("BPARating").filter(pl.col("BPARating").is_not_null(), pl.col("BPARating").is_between(1, 5)).cast(pl.Int32).value_counts().map_elements(lambda x: struct_to_str(x, "BPARating")).alias("BPARatingBreakdown"),
    
    # Neither
    num_w_percentage_expr(((pl.col("offeredPEA").is_in(["No, not offered PEA", "Yes, don't want"])).or_(pl.col("hadPEA") == "No")).and_(pl.col("hadBPA") == "No BPA"), "Have not had either PEA or BPA"),
).sort("region").vstack(
    pl.DataFrame({
        "region": ["Total"],
        "Number of individuals": [f"{len(g4.filter(pl.col('offeredPEA').is_not_null()))} (100%)"],

        "Single": [num_w_percentage_float(g4.select(pl.col("Number of PH medical therapies (vasodilators)") == 1).sum().item(), len(g4))],
        "Dual": [num_w_percentage_float(g4.select(pl.col("Number of PH medical therapies (vasodilators)") == 2).sum().item(), len(g4))],
        "Triple": [num_w_percentage_float(g4.select(pl.col("Number of PH medical therapies (vasodilators)") == 3).sum().item(), len(g4))],

        "Taking parenteral drugs": [num_w_percentage_float(g4.select(pl.col("takingParenteralDrugs")).sum().item(), len(g4))],

        "Unlicensed/trial drugs": [f"{g4['Unlicensed/trial drugs'].sum()} ({round(g4['Unlicensed/trial drugs'].sum() / len(g4) * 100, 1)}%)"],

        "Calcium channel blockers": [
            num_w_percentage_float(
                g4.select(pl.col("namesOfMedications").str.contains(
                    r"Amlodipine \(Istin\)|Diltiazem \(Adizem, Angitil, Dilcardia, Retalzem, Slozem, Tildiem, Uard, Viazem, Zemtard\)|Nifedipine \(Adalat, Adanif, Adipine, Coracten, Dexipress, Fortipine, Neozipine, Nidef, Nifedipress, Tensipine, Valni\)|Nircadipine \(Cardene\)"
                )).sum().item(),
                len(g4),
            ),
        ],

        "VKA": [num_w_percentage_float(g4.select(pl.col("antiCoagulants").str.contains(r"Warfarin or Acenocoumarol tablets \- these require regular blood tests to check the dose")).sum().item(), len(g4))],
        "DOAC": [num_w_percentage_float(g4.select(pl.col("antiCoagulants").str.contains(r"Dabigatran, rivaroxaban, apixaban, edoxaban or betrixaban tablets")).sum().item(), len(g4))],
        "LMWH": [num_w_percentage_float(g4.select(pl.col("antiCoagulants").str.contains(r"Injections under the skin \(e.g. heparin drugs\) - there are lots of types")).sum().item(), len(g4))],

        "Recieved PEA":[num_w_percentage_float((g4["hadPEA"] == "Yes").sum(), len(g4))],
        "PEA Rating":[f"{round(g4.filter(pl.col('PEARating').is_not_null(), pl.col('PEARating').is_between(1, 5))['PEARating'].cast(pl.Int32).mean(), 1)} ({round(g4.filter(pl.col('PEARating').is_not_null(), pl.col('PEARating').is_between(1, 5))['PEARating'].cast(pl.Int32).std(), 1)})"],
        "PEARatingBreakdown": [str(g4.filter(pl.col('PEARating').is_not_null(), pl.col('PEARating').is_between(1, 5))['PEARating'].cast(pl.Int32).value_counts())],

        "Had BPA":[num_w_percentage_float(g4["hadBPA"].str.contains("Yes BPA").sum(), len(g4))],
        "Sessions of BPA": [struct_to_str(g4.filter(pl.col("sessionsOfBPA").is_not_null(), pl.col("sessionsOfBPA").str.contains("[-More]"))["sessionsOfBPA"].value_counts().to_struct(), "sessionsOfBPA")],
        "Mean BPA Rating":[f"{round(g4.filter(pl.col('BPARating').is_not_null(), pl.col('BPARating').is_between(1, 5))['BPARating'].cast(pl.Int32).mean(), 1)} ({round(g4.filter(pl.col('BPARating').is_not_null(), pl.col('BPARating').is_between(1, 5))['BPARating'].cast(pl.Int32).std(), 1)})"],
        "BPARatingBreakdown": [str(g4.filter(pl.col('BPARating').is_not_null(), pl.col('BPARating').is_between(1, 5))['BPARating'].cast(pl.Int32).value_counts())],

        "Have not had either PEA or BPA":[num_w_percentage_float(len(g4.filter((pl.col("offeredPEA").is_in(["No, not offered PEA", "Yes, don't want"]).or_(pl.col("hadPEA") == "No").and_(pl.col("hadBPA") == "No BPA")))), len(g4))],
    })
)



# Supplementary: Groups 2 and 3

Paper p3. Additional analysis for the underrepresented PH groups 2 and 3.

## Demographics by PH group

Age, sex, and diagnosis distribution across all PH groups.

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
)["diagnosisClassification (cleaned)"].value_counts().sort(by="diagnosisClassification (cleaned)").select(
    pl.col("diagnosisClassification (cleaned)"),
    pl.col("count"),
    (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
).group_by("diagnosisClassification (cleaned)").agg(
    pl.col("age").mean().round(1).alias("Mean age"),
    pl.col("age").std().round(1).alias("Standard deviation of age"),
    pl.col("sexAtBirth").value_counts().sort().map_elements(lambda x: struct_to_str(x, "sexAtBirth")),
).sort(by="diagnosisClassification (cleaned)")


## Group 2 – Medications

Medication usage for Group 2 (PH associated with left heart disease).

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("diagnosisClassification (cleaned)") == "Group 2 - PH associated with left heart disease",
).select(
    (pl.col("namesOfMedications").is_not_null() | pl.col("namesOfMedicationsOther").is_not_null()).value_counts(),
).select(
    pl.col("namesOfMedications").struct.field("namesOfMedications"),
    pl.col("namesOfMedications").struct.field("count"),
    (pl.col("namesOfMedications").struct.field("count") / pl.col("namesOfMedications").struct.field("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("diagnosisClassification (cleaned)") == "Group 2 - PH associated with left heart disease",
).select(
    pl.concat_str(pl.col("namesOfMedications"), pl.col("namesOfMedicationsOther"), separator=";", ignore_nulls=True).str.split(";").explode(),
).filter(
    pl.col("namesOfMedications").is_not_null(),
    pl.col("namesOfMedications").str.len_chars() > 0,
).select(
    pl.col("namesOfMedications").value_counts(),
).with_columns(
    pl.col("namesOfMedications").struct.field("namesOfMedications"),
    pl.col("namesOfMedications").struct.field("count"),
    (pl.col("namesOfMedications").struct.field("count") / pl.col("namesOfMedications").struct.field("count").sum() * 100).round(1).alias("Percentage"),
).filter(
    pl.col("count") > 3,
).sort(by="count", descending=True)


## Group 3 – Medications

Medication usage for Group 3 (PH associated with lung disease).

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("diagnosisClassification (cleaned)") == "Group 3 - PH associated with lung disease (Including COPD, Interstitial/fibrotic lung disease, other lung diseases)",
).select(
    (pl.col("namesOfMedications").is_not_null() | pl.col("namesOfMedicationsOther").is_not_null()).value_counts(),
).select(
    pl.col("namesOfMedications").struct.field("namesOfMedications"),
    pl.col("namesOfMedications").struct.field("count"),
    (pl.col("namesOfMedications").struct.field("count") / pl.col("namesOfMedications").struct.field("count").sum() * 100).round(1).alias("Percentage"),
)

In [ ]:
adult_df.filter(
    pl.col("Are you answering this questionnaire as a... (cleaned)") != "Parent or guardian of a child with pulmonary hypertension (PH)",
    pl.col("diagnosisClassification (cleaned)") == "Group 3 - PH associated with lung disease (Including COPD, Interstitial/fibrotic lung disease, other lung diseases)",
).select(
    pl.concat_str(pl.col("namesOfMedications"), pl.col("namesOfMedicationsOther"), separator=";", ignore_nulls=True).str.split(";").explode(),
).filter(
    pl.col("namesOfMedications").is_not_null(),
    pl.col("namesOfMedications").str.len_chars() > 0,
).select(
    pl.col("namesOfMedications").value_counts(),
).with_columns(
    pl.col("namesOfMedications").struct.field("namesOfMedications"),
    pl.col("namesOfMedications").struct.field("count"),
    (pl.col("namesOfMedications").struct.field("count") / pl.col("namesOfMedications").struct.field("count").sum() * 100).round(1).alias("Percentage"),
).filter(
    pl.col("count") > 3,
).sort(by="count", descending=True)
